# TAE-IA · Module 6 · L14 — The Frequency World: FFT and Spectrograms

| | |
|---|---|
| **Module** | 6 — Practical AI Applications: Images and Audio |
| **Lesson** | L14 |
| **Track** | B — Audio |
| **Runtime** | CPU is sufficient — no GPU required |
| **Drive quota** | No large model downloads — no cleanup needed |

## Learning objectives

By the end of this notebook you will be able to:
1. Compute and plot the FFT power spectrum of a signal and identify peaks by frequency
2. Explain the difference between FFT (global) and STFT (windowed) representations
3. Generate STFT spectrograms with `librosa.stft()` and `librosa.display.specshow()`
4. Quantify how `n_fft` and `hop_length` control frequency vs. time resolution
5. Read a spectrogram and identify phonetic and musical events by their spectral shape

---

## Cell 0 — Setup

In [ ]:
# ================================================================
# TAE-IA M6 · Standard Setup -- DO NOT MODIFY THIS CELL
# ================================================================
import os, sys, random
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

MODEL_CACHE = '/content/drive/MyDrive/TAE_IA_M6/models'
os.makedirs(MODEL_CACHE, exist_ok=True)

os.environ['HF_HOME']            = MODEL_CACHE
os.environ['TORCH_HOME']         = MODEL_CACHE
os.environ['TRANSFORMERS_CACHE'] = os.path.join(MODEL_CACHE, 'hub')

SEED = 42
random.seed(SEED); np.random.seed(SEED)

print('Setup complete. CPU runtime is sufficient for L14.')

## Cell 1 — Install and import

In [ ]:
!pip install librosa soundfile -q

import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import IPython.display as ipd

print(f'librosa {librosa.__version__}')

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Cell 2 — Regenerate L13 audio signals

We regenerate the same three synthetic signals from L13 so this notebook is self-contained.

In [ ]:
SR  = 22050
DUR = 3.0
N   = int(SR * DUR)
t   = np.linspace(0, DUR, N, endpoint=False)

# Voice-like: harmonic series + syllable envelope + silence gaps
f0 = 150.0
y_voice = sum((1.0 / k) * np.sin(2 * np.pi * f0 * k * t) for k in range(1, 8))
env = 0.5 + 0.5 * np.sin(2 * np.pi * 5 * t)
gap = np.ones(N)
gap[int(0.3*SR):int(0.5*SR)] = 0.0
gap[int(1.8*SR):int(2.0*SR)] = 0.0
y_voice = (y_voice * env * gap).astype(np.float32)
y_voice /= np.abs(y_voice).max() + 1e-8

# Music-like: A-major chord with tremolo
y_music = sum(np.sin(2 * np.pi * f * t) for f in [440.0, 554.37, 659.25])
y_music = (y_music * (1.0 + 0.3 * np.sin(2 * np.pi * 6 * t))).astype(np.float32)
y_music /= np.abs(y_music).max() + 1e-8

# Ambient-like: band-limited pink noise
rng   = np.random.default_rng(SEED)
white = rng.normal(0, 1, N).astype(np.float32)
fft_w = np.fft.rfft(white)
freqs = np.fft.rfftfreq(N, d=1.0/SR)
band  = ((freqs >= 200) & (freqs <= 4000)).astype(float)
with np.errstate(divide='ignore', invalid='ignore'):
    pink = np.where(freqs > 200, 200.0 / freqs, 1.0)
y_ambient = np.fft.irfft(fft_w * band * pink, n=N).astype(np.float32)
y_ambient /= np.abs(y_ambient).max() + 1e-8

AUDIO = {'voice': (y_voice, SR), 'music': (y_music, SR), 'ambient': (y_ambient, SR)}
COLOURS = {'voice': '#2C75FF', 'music': '#27ae60', 'ambient': '#8e44ad'}

print('Audio regenerated.')
for name, (y, sr) in AUDIO.items():
    print(f'  {name}: {len(y)} samples, {len(y)/sr:.1f}s')

---
## Section 2.1 — FFT Power Spectrum

The FFT takes the **entire signal** and produces a list of frequency components.

**Key concepts:**
- `np.fft.rfft(y)` — real FFT: returns `N//2 + 1` complex values (positive frequencies only)
- Magnitude: `np.abs(fft_result)` — how much of each frequency is present
- Frequency axis: `np.fft.rfftfreq(N, d=1/sr)` — maps each bin index to Hz
- Frequency resolution: `sr / N` Hz per bin — finer with more samples
- **Always apply a window function** (e.g., Hanning) before FFT to reduce *spectral leakage* — the smearing of energy from one frequency bin into adjacent bins.

In [ ]:
# First: intuition with a pure 440 Hz tone
t_1s  = np.linspace(0, 1.0, SR, endpoint=False)
y_440 = np.sin(2 * np.pi * 440 * t_1s).astype(np.float32)

window = np.hanning(len(y_440))
fft_   = np.fft.rfft(y_440 * window)
mag    = np.abs(fft_)
freq   = np.fft.rfftfreq(len(y_440), d=1.0/SR)
mag_db = librosa.amplitude_to_db(mag, ref=mag.max())

freq_res = SR / len(y_440)
print(f'Signal: 1s at {SR} Hz → {len(y_440)} samples')
print(f'FFT output: {len(mag)} bins, frequency resolution = {freq_res:.2f} Hz/bin')
print(f'Peak bin index: {np.argmax(mag)}  → frequency: {freq[np.argmax(mag)]:.1f} Hz')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(freq, mag_db, color='#27ae60', linewidth=0.9)
ax.axvline(440, color='red', linewidth=1.5, linestyle='--', label='440 Hz (A4)')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('Amplitude (dB)')
ax.set_title('FFT power spectrum — pure 440 Hz tone', fontweight='bold')
ax.set_xlim(0, 2000)
ax.set_ylim(-80, 5)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def plot_fft(y, sr, title, ax, color='#27ae60', xlim=5000):
    """Plot FFT power spectrum with Hanning window."""
    N      = len(y)
    window = np.hanning(N)
    fft_   = np.fft.rfft(y * window)
    mag    = np.abs(fft_)
    freq   = np.fft.rfftfreq(N, d=1.0/sr)
    mag_db = librosa.amplitude_to_db(mag, ref=mag.max())

    ax.plot(freq, mag_db, color=color, linewidth=0.7)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('Amplitude (dB)')
    ax.set_xlim(0, xlim)
    ax.set_ylim(-80, 5)
    ax.axhline(-60, color='gray', linewidth=0.5, linestyle=':', alpha=0.6)

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

plot_fft(y_voice,   SR, 'VOICE — harmonic series (fundamental 150 Hz + overtones)', axes[0], COLOURS['voice'])
plot_fft(y_music,   SR, 'MUSIC — A-major chord (440, 554, 659 Hz + overtones)',     axes[1], COLOURS['music'])
plot_fft(y_ambient, SR, 'AMBIENT — band-limited noise (200–4000 Hz)',                axes[2], COLOURS['ambient'])

# Annotate chord peaks
for f, label in [(440, 'A4'), (554, 'C#5'), (659, 'E5')]:
    axes[1].axvline(f, color='#c0392b', linewidth=1.0, linestyle='--', alpha=0.6)
    axes[1].text(f + 10, -5, label, color='#c0392b', fontsize=9)

plt.suptitle('FFT power spectra — voice · music · ambient', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 2.2 — STFT Spectrograms

The STFT applies the FFT to **short overlapping windows** of the signal, producing a 2D matrix:
- Rows = frequency bins (0 to Nyquist)
- Columns = time frames
- Values = complex STFT coefficients; we take the magnitude and convert to dB

**Always convert to dB before plotting** — the raw linear magnitude compresses 99% of the dynamic range into a tiny bright band at the top.

In [ ]:
# Compute STFT for all three signals with default parameters
N_FFT   = 2048
HOP_LEN = 512

spectrograms = {}
for name, (y, sr) in AUDIO.items():
    D = librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LEN, window='hann')
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
    spectrograms[name] = S_db

    freq_res = sr / N_FFT
    time_res = HOP_LEN / sr * 1000
    n_frames = D.shape[1]
    print(f'{name}: shape={D.shape}  freq_res={freq_res:.1f} Hz/bin  '
          f'time_res={time_res:.1f} ms/frame  n_frames={n_frames}')

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

for ax, (name, (y, sr)) in zip(axes, AUDIO.items()):
    S_db = spectrograms[name]
    img  = librosa.display.specshow(
        S_db, sr=sr, hop_length=HOP_LEN,
        x_axis='time', y_axis='hz',
        ax=ax, cmap='magma'
    )
    ax.set_title(f'{name.upper()}  ·  n_fft={N_FFT}, hop={HOP_LEN}  '
                 f'·  shape={S_db.shape}',
                 fontsize=11, fontweight='bold')
    ax.set_ylabel('Frequency (Hz)')
    plt.colorbar(img, ax=ax, format='%+2.0f dB', pad=0.01)

axes[-1].set_xlabel('Time (s)')
plt.suptitle('STFT Spectrograms — voice · music · ambient  (n_fft=2048, hop=512)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### What to look for in the spectrograms:

| Signal | Characteristic pattern | Reason |
|---|---|---|
| **Voice** | Horizontal bright bands (formants) with vertical dark gaps | Formants = vowel resonances; gaps = silence between syllables |
| **Music** | Horizontal stripes at fixed frequencies with harmonics above | Chord frequencies + their overtone series; tremolo = amplitude modulation |
| **Ambient** | Uniform texture — energy spread across frequencies and time | Noise has no dominant frequency or time structure |

---
## Section 2.3 — `n_fft` Sweep: Observing the Resolution Trade-off

The same voice clip rendered with three values of `n_fft`. The `hop_length` is fixed so time resolution only changes through the window length effect.

**What to expect:**
- `n_fft=256`: blurry frequency axis (86 Hz/bin), sharp time axis — good for transients
- `n_fft=1024`: balanced — the general-purpose default
- `n_fft=4096`: sharp frequency axis (5.4 Hz/bin), smeared time axis — good for pitch

In [ ]:
n_fft_values = [256, 1024, 4096]
HOP_FIXED    = 256
y_v, sr_v    = AUDIO['voice']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, n_fft in zip(axes, n_fft_values):
    D    = librosa.stft(y_v, n_fft=n_fft, hop_length=HOP_FIXED, window='hann')
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    freq_res = sr_v / n_fft
    time_res = HOP_FIXED / sr_v * 1000
    n_bins   = D.shape[0]

    img = librosa.display.specshow(
        S_db, sr=sr_v, hop_length=HOP_FIXED,
        x_axis='time', y_axis='hz',
        ax=ax, cmap='magma'
    )
    ax.set_title(
        f'n_fft = {n_fft}\n'
        f'freq res: {freq_res:.1f} Hz/bin  ·  {n_bins} bins\n'
        f'time res: {time_res:.1f} ms/frame',
        fontsize=10, fontweight='bold'
    )
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    plt.colorbar(img, ax=ax, format='%+2.0f dB', pad=0.01)

plt.suptitle(f'n_fft sweep (hop_length={HOP_FIXED} fixed) — Voice signal',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Resolution table for all common n_fft values at 22050 Hz
print(f'{"n_fft":>8}  {"freq res (Hz/bin)":>20}  {"#bins":>7}  {"window len (ms)":>18}')
print('-' * 62)
for n in [128, 256, 512, 1024, 2048, 4096]:
    freq_res = SR / n
    win_ms   = n / SR * 1000
    n_bins   = n // 2 + 1
    marker   = ' ← Whisper default' if n == 400 else (
               ' ← music default'   if n == 2048 else '')
    print(f'{n:>8}  {freq_res:>20.2f}  {n_bins:>7}  {win_ms:>18.1f}{marker}')

---
## Section 2.4 — `hop_length` Sweep

Now fix `n_fft` and vary `hop_length`. This changes **how many time frames** appear in the spectrogram without changing frequency resolution.

In [ ]:
hop_values = [128, 512, 2048]
N_FFT_FIXED = 2048

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, hop in zip(axes, hop_values):
    D    = librosa.stft(y_v, n_fft=N_FFT_FIXED, hop_length=hop, window='hann')
    S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)

    time_res = hop / sr_v * 1000
    n_frames = D.shape[1]
    overlap  = (1 - hop / N_FFT_FIXED) * 100

    img = librosa.display.specshow(
        S_db, sr=sr_v, hop_length=hop,
        x_axis='time', y_axis='hz',
        ax=ax, cmap='magma'
    )
    ax.set_title(
        f'hop = {hop}\n'
        f'time res: {time_res:.1f} ms  ·  {n_frames} frames\n'
        f'overlap: {overlap:.0f}%',
        fontsize=10, fontweight='bold'
    )
    ax.set_ylabel('Frequency (Hz)')
    ax.set_xlabel('Time (s)')
    plt.colorbar(img, ax=ax, format='%+2.0f dB', pad=0.01)

plt.suptitle(f'hop_length sweep (n_fft={N_FFT_FIXED} fixed) — Voice signal',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 2.5 — Waveform + Spectrogram Side-by-Side

Aligning the waveform and spectrogram in time makes it easier to see which waveform events correspond to which spectral patterns.

In [ ]:
fig, (ax_wave, ax_spec) = plt.subplots(2, 1, figsize=(14, 7),
                                         gridspec_kw={'height_ratios': [1, 2]})

# Waveform
librosa.display.waveshow(y_v, sr=SR, ax=ax_wave, color=COLOURS['voice'], alpha=0.85)
ax_wave.set_title('Voice — waveform', fontweight='bold')
ax_wave.set_xlabel('')
ax_wave.set_ylabel('Amplitude')

# Spectrogram
D    = librosa.stft(y_v, n_fft=2048, hop_length=512)
S_db = librosa.amplitude_to_db(np.abs(D), ref=np.max)
img  = librosa.display.specshow(S_db, sr=SR, hop_length=512,
                                  x_axis='time', y_axis='hz',
                                  ax=ax_spec, cmap='magma')
ax_spec.set_title('Voice — STFT spectrogram (n_fft=2048, hop=512)', fontweight='bold')
ax_spec.set_ylabel('Frequency (Hz)')
plt.colorbar(img, ax=ax_spec, format='%+2.0f dB', pad=0.01)

# Align x-axis
ax_wave.set_xlim(0, DUR)
ax_spec.set_xlim(0, DUR)

plt.suptitle('Waveform vs. spectrogram — same signal, two views', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nObserve: the silence gaps in the waveform (flat near zero) appear as dark vertical'
      ' bands in the spectrogram.')

---
## Exercise 1 — Multi-tone identification

Generate a signal that contains **two different tones in sequence**: 330 Hz for the first 1.5 seconds, then 660 Hz for the last 1.5 seconds.

1. Compute the global FFT. Can you distinguish the two tones?
2. Compute the STFT spectrogram. Can you distinguish the two tones?
3. Explain in a markdown cell why the two representations give different results for this signal.

This exercise directly motivates the STFT: the FFT cannot answer "when did the pitch change?"

In [ ]:
# Two-tone signal: 330 Hz → 660 Hz
t_half = np.linspace(0, 1.5, int(SR * 1.5), endpoint=False)
y_low  = np.sin(2 * np.pi * 330 * t_half).astype(np.float32)
y_high = np.sin(2 * np.pi * 660 * t_half).astype(np.float32)
y_two_tones = np.concatenate([y_low, y_high])

# Play it
ipd.display(ipd.Audio(y_two_tones, rate=SR))

# TODO: compute and plot both FFT and STFT for y_two_tones
# Then answer Q3 in a markdown cell below

**Exercise 1 — Answer:**

[YOUR ANSWER — explain why FFT cannot separate the two tones in time, but STFT can]

---
## Exercise 2 — Optimal parameters for a speech task

Whisper uses `n_fft=400` and `hop_length=160` at 16000 Hz.

1. Compute the frequency resolution and time resolution for Whisper's parameters
2. Resampled `y_voice` to 16000 Hz and plot its spectrogram using Whisper's exact parameters
3. In a markdown cell: explain why Whisper chose a relatively small `n_fft=400` (25 ms window) instead of the music default of 2048 (~93 ms). What acoustic property of speech makes 25 ms appropriate?

In [ ]:
# Whisper parameters
N_FFT_WHISPER  = 400
HOP_WHISPER    = 160
SR_WHISPER     = 16000

freq_res = SR_WHISPER / N_FFT_WHISPER
time_res = HOP_WHISPER / SR_WHISPER * 1000
print(f'Whisper STFT parameters:')
print(f'  n_fft={N_FFT_WHISPER}  hop={HOP_WHISPER}  sr={SR_WHISPER}')
print(f'  Frequency resolution : {freq_res:.1f} Hz/bin')
print(f'  Time resolution      : {time_res:.1f} ms/frame')
print(f'  Window duration      : {N_FFT_WHISPER/SR_WHISPER*1000:.1f} ms')

# TODO: resample y_voice to 16000 Hz and plot spectrogram with Whisper parameters

**Exercise 2 — Answer:**

[YOUR ANSWER — why 25 ms window for speech recognition?]

---
## Part 4 — Critical Analysis

Answer all four questions with real outputs from the cells above.

### Q1 — FFT and ZCR

From your Section 2.1 FFT spectra: which of the three signals has the highest ZCR (recall from L13)? Does the FFT spectrum visually confirm why? Explain the connection between ZCR and spectral flatness in 2–3 sentences.

**[YOUR ANSWER]**

---

### Q2 — Spectrogram patterns

In your Section 2.2 spectrograms, identify **one feature** that appears in the voice spectrogram but not in the ambient spectrogram. Name:
- The approximate frequency range it occupies (Hz)
- What acoustic event it represents
- Whether you could detect that event from the waveform alone

**[YOUR ANSWER]**

---

### Q3 — n_fft trade-off for speech recognition

From your Section 2.3 n_fft sweep: comparing `n_fft=256` vs. `n_fft=4096` for the voice signal — for a Whisper-like speech recognition task, which would you choose and why?

Consider: speech phonemes have typical durations of 50–200 ms; the fundamental frequency of speech is 80–300 Hz; consonants (stops, fricatives) have durations of 5–50 ms.

**[YOUR ANSWER]** *(~4 sentences)*

---

### Q4 — Bridge to AI: spectrograms as images

Explain in 3 sentences:
1. Why a CNN trained on natural images (CIFAR-10) could in principle classify spectrograms without retraining from scratch
2. What assumption this transfer requires
3. One specific scenario where this assumption would break down

**[YOUR ANSWER]**

---

---
## Submission Checklist

Before saving and submitting:

- [ ] Cell 0 ran without errors (Drive mounted)
- [ ] Section 2.1: FFT spectra plotted for all 3 audio types with peaks labelled
- [ ] Section 2.2: STFT spectrograms plotted for all 3 audio types
- [ ] Section 2.3: n_fft sweep (256 / 1024 / 4096) plotted side-by-side
- [ ] Section 2.4: hop_length sweep plotted
- [ ] Section 2.5: waveform + spectrogram aligned side-by-side
- [ ] Exercise 1 complete (two-tone FFT vs. STFT + written explanation)
- [ ] Exercise 2 complete (Whisper parameters + written explanation)
- [ ] Critical Analysis Q1–Q4 answered (no `[YOUR ANSWER]` remaining)
- [ ] Notebook saved to Drive

**No Drive cleanup needed after L14** — no large models were downloaded.

---
*TAE-IA 2025 · Cocyten-Nayarit · Module 6 · L14*  
*Track B — Audio | Next: L15 — Mel, MFCC, and the Language Models Understand*